# Miller OTA autoresearch — eda-agents analog loop demo

Walks the analog loop from a natural-language spec to a sized Miller OTA on **IHP SG13G2**,
using the `AutoresearchRunner` greedy loop + ngspice + gm/ID LUTs.

Companion to the slide deck at `../claude_design_slides/`.

**Prerequisites (host-side)**

- You cloned `eda-agents` — this notebook lives at `tutorials/agents-analog-digital/demo/`.
- You are running Jupyter from an **activated venv** (Python 3.9+).
- `ngspice` is installed and on PATH.
- `$PDK_ROOT` points at an IHP-Open-PDK tree with `ihp-sg13g2/` present.
- `$EDA_AGENTS_IHP_LUT_DIR` points at your `ihp-gmid-kit` clone (LUT source).
- `$OPENROUTER_API_KEY` **or** `$ZAI_API_KEY` is set (LLM proposal model).

All `RUN_*` flags default to `False` — opening the notebook is a safe no-op.
Flip them once the environment check passes.

## Step 0 — Editable install from the repo root

The analog loop depends on the `eda_agents` Python package. From a fresh venv, install it
in editable mode so the notebook sees your local checkout.

Flip `RUN_PIP_INSTALL` to `True` the **first** time you run this notebook in a new venv.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

RUN_PIP_INSTALL   = False         # flip once per venv
RUN_AUTORESEARCH  = False         # flip after Step 3 succeeds
AUTORESEARCH_BUDGET = 6           # SPICE evals; ~2-3 min on a laptop
LLM_MODEL         = "zai/GLM-4.5-Flash"

REPO_ROOT = Path.cwd().resolve().parents[2]   # demo/ -> agents-analog-digital/ -> tutorials/ -> eda-agents/
WORK_DIR  = Path.cwd() / "autoresearch_miller_ota"

print(f"Repo root : {REPO_ROOT}")
print(f"Python    : {sys.executable}")
print(f"venv      : {os.environ.get('VIRTUAL_ENV', 'UNSET (activate a venv first)')}")

if RUN_PIP_INSTALL:
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_ROOT)], check=True)
else:
    print("[rehearse] RUN_PIP_INSTALL=False — skipping install.")

## Step 1 — Environment check

Verify ngspice, the PDK, the gm/ID LUT clone, and the LLM key. This cell makes
**no changes** — it just prints what's wired up.

In [ ]:
import shutil

ng = shutil.which("ngspice")
pdk_root = os.environ.get("PDK_ROOT", "")
lut_dir  = os.environ.get("EDA_AGENTS_IHP_LUT_DIR", "")

print(f"ngspice on PATH        : {ng or 'MISSING'}")
print(f"PDK_ROOT               : {pdk_root or 'UNSET'}")
if pdk_root:
    ihp_models = Path(pdk_root) / "ihp-sg13g2" / "libs.tech" / "ngspice" / "models"
    print(f"IHP models present     : {ihp_models.is_dir()}")
print(f"EDA_AGENTS_IHP_LUT_DIR : {lut_dir or 'UNSET (required for Miller OTA)'}")

for key in ("OPENROUTER_API_KEY", "ZAI_API_KEY"):
    print(f"{key:<22}: {'set' if os.environ.get(key) else 'unset'}")

## Step 2 — Instantiate the Miller OTA `CircuitTopology`

All analog topologies implement the `CircuitTopology` ABC — `design_space()`,
`params_to_sizing()`, `generate_netlist()`, `compute_fom()`, `check_validity()`.

The agent harness never hardcodes a topology name — it derives prompts and tool
specs from these metadata methods.

In [ ]:
from eda_agents.topologies.ota_miller import MillerOTATopology

topo = MillerOTATopology()
print(f"topology_name  : {topo.topology_name()}")
print(f"pdk            : {topo.pdk}")
print(f"design_space   : {list(topo.design_space().keys())}")
print("default_params :")
for k, v in topo.default_params().items():
    print(f"  {k} = {v}")
print(f"specs          : {topo.specs_description()}")

## Step 3 — One SPICE eval at default params

Cheap sanity check: generates the ngspice deck, runs it, parses the
measurements. If this fails, autoresearch will fail the same way —
investigate the PDK config before flipping `RUN_AUTORESEARCH`.

In [ ]:
import tempfile
from eda_agents.core.spice_runner import SpiceRunner

runner = SpiceRunner(pdk=topo.pdk)
missing = runner.validate_pdk()
print(f"PDK problems: {missing or 'none'}")

if not missing:
    params = topo.default_params()
    sizing = topo.params_to_sizing(params)
    cir    = topo.generate_netlist(sizing, Path(tempfile.mkdtemp()))
    result = runner.run(cir)
    if result.success:
        fom = topo.compute_fom(result, sizing)
        print(f"Adc = {result.Adc_dB:.1f} dB")
        print(f"GBW = {result.GBW_Hz/1e6:.2f} MHz")
        print(f"PM  = {result.PM_deg:.1f} deg")
        print(f"FoM = {fom:.3e}")
    else:
        print(f"SPICE failed: {result.error}")

## Step 4 — Run the autoresearch loop

The runner:

1. Reads `program.md` (empty on first iteration).
2. Asks the LLM for the next parameter proposal.
3. Maps params → sizing → netlist, runs SPICE, scores FoM.
4. Appends a row to `results.tsv`.
5. Updates `program.md` with the learned insight + new best.

Budget = 6 evaluations ≈ 2–3 minutes. Crashed run? Re-run this cell —
the runner reads its own prior state back.

In [ ]:
import asyncio
import json
from eda_agents.agents.autoresearch_runner import AutoresearchRunner

async def _run():
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    r = AutoresearchRunner(topology=topo, model=LLM_MODEL, budget=AUTORESEARCH_BUDGET)
    return await r.run(WORK_DIR)

if RUN_AUTORESEARCH:
    result = asyncio.get_event_loop().run_until_complete(_run())
    print(result.summary)
    print(f"validity_rate : {result.validity_rate:.0%}")
    print(f"tsv_path      : {result.tsv_path}")
    if result.best_valid:
        print("best_params   :", json.dumps(result.best_params, indent=2))
else:
    print("[rehearse] RUN_AUTORESEARCH=False — skipping the real loop.")
    print(f"When enabled, program.md + results.tsv are written under {WORK_DIR}")

## Step 5 — Inspect `program.md` + `results.tsv`

These two files are the entire persistence layer. The LLM reads them on the
next run to resume without replaying the full prompt history.

In [ ]:
prog = WORK_DIR / "program.md"
tsv  = WORK_DIR / "results.tsv"

if prog.exists():
    print(f"--- {prog} ---\n")
    print(prog.read_text()[:2000])
else:
    print(f"{prog} not yet written.")

if tsv.exists():
    rows = tsv.read_text().splitlines()
    print(f"\n--- {tsv} (head + tail) ---")
    for r in rows[:1] + rows[-10:]:
        print(r)

## Cleanup

```bash
rm -rf autoresearch_miller_ota/
```

Leave the files in place if you want to resume later — the runner picks up
from the last `results.tsv` row.

## Next

- Digital-loop companion: `./agents_rtl2gds_counter.ipynb` drives LibreLane
  through the `project_manager` + 4 specialists hierarchy on a 4-bit counter.
- Slide deck: `../claude_design_slides/AI Agents for Analog and Digital IC Design.html`.
- For a different topology, swap in `GF180OTATopology` or `AnalogAcademyOTATopology`.
- Source of truth for agent + skill names: `.claude/agents/*.md` and
  `src/eda_agents/skills/{analog,digital}.py`.